## 导入依赖

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 设置绘图风格
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # 支持中文字体
plt.rcParams['axes.unicode_minus'] = False

## 加载推理结果

In [ ]:
# 假设 run_benchmark.py 生成了这个 debug 文件
RESULT_FILE = "../data/results/predictions_debug.json" 

try:
    with open(RESULT_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"成功加载 {len(raw_data)} 条预测结果")
except FileNotFoundError:
    print(f"文件未找到: {RESULT_FILE}。请先运行 evaluation/run_benchmark.py")
    raw_data = []

## 数据展平

In [ ]:
# 我们需要将 (Aspect, Sentiment) 对拆开，以便计算混淆矩阵
# 因为一个句子可能包含多个方面，我们需要逐个对齐

y_true = []
y_pred = []
labels = ["positive", "negative", "neutral"]

# 简单的对齐逻辑：
# 这里为了简化混淆矩阵的绘制，我们只统计“命中”的方面的情感分类准确率。
# (注意：完整的 F1 计算在 compute_metrics.py 中已经做得更严谨了，这里主要为了看情感偏差)

for item in raw_data:
    gold_aspects = {a['aspect_term'].lower(): a['sentiment'] for a in item['gold']}
    pred_aspects = {a['aspect_term'].lower(): a['sentiment'] for a in item['pred']}
    
    # 找到共有的方面（交集）来对比情感判断
    common_terms = set(gold_aspects.keys()) & set(pred_aspects.keys())
    
    for term in common_terms:
        y_true.append(gold_aspects[term])
        y_pred.append(pred_aspects[term])

print(f"共提取出 {len(y_true)} 个匹配的方面用于情感混淆分析")

## 混淆矩阵可视化

In [ ]:
if y_true:
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('预测情感 (Predicted)')
    plt.ylabel('真实情感 (True)')
    plt.title('情感分类混淆矩阵 (基于正确提取的方面)')
    plt.show()

    # 打印分类报告
    print(classification_report(y_true, y_pred, target_names=labels))
else:
    print("没有匹配的数据可供绘图。")

## 错误分析

In [ ]:
# 让我们看看模型在哪些具体的句子上“思考”错了
# 重点关注：Aspect 提取正确，但情感判断错误的例子

print("=== 典型错误样本分析 ===")

error_count = 0
for item in raw_data:
    gold_aspects = {a['aspect_term'].lower(): a['sentiment'] for a in item['gold']}
    pred_aspects = {a['aspect_term'].lower(): a['sentiment'] for a in item['pred']}
    
    common_terms = set(gold_aspects.keys()) & set(pred_aspects.keys())
    
    for term in common_terms:
        if gold_aspects[term]!= pred_aspects[term]:
            error_count += 1
            if error_count <= 5: # 只打印前 5 个错误
                print(f"\n原文: {item['text']}")
                print(f"方面: {term}")
                print(f"❌ 预测: {pred_aspects[term]} | ✅ 真实: {gold_aspects[term]}")
                print(f"思考过程片段 (CoT):")
                # 截取思考过程的最后一部分，通常包含结论
                think_content = item.get('think', '')
                print(f"{think_content[-200:]}..." if len(think_content) > 200 else think_content)
                print("-" * 50)